# Chapter 12.2. 선호 기반 보상모델 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter12_2_prefer_reward.ipynb)

책 본문: [Chapter 12](https://smhanlab.com/book-ml/kor/ml2/chapter12.html)

이 노트북은 책 12.2절의 장난감 예제를 그대로 실행합니다. 사람은
겉으로 드러나지 않는 진짜 보상함수 $r^*(\text{traj}) = 2\cdot\text{부드러움} - 1\cdot\text{덜컹거림}$
을 마음속으로 갖고 **두 궤적 중 어느 쪽이 좋은지만** 비교해 라벨을 달고,
학습 알고리즘은 이 진짜 보상함수를 전혀 모른 채 선호 쌍 300개만으로
Bradley-Terry 모델로 보상모델 $r_\phi$의 가중치를 복원합니다.

## 1. 설정: 숨겨진 진짜 보상과 선호 쌍 생성

궤적을 (부드러움, 덜컹거림) 2차원 좌표로 요약합니다. '사람'은 진짜
보상 $r^*$로 두 궤적을 비교해 더 나은 쪽을 win으로 라벨링하고,
학습 알고리즘은 $(win, lose)$ 쌍만 받습니다 — $r^*$의 수식은 절대 안 줍니다.

In [1]:
import math
import random

import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt

IMG = "/home/smhan/book-ml/kor/src/images"
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, z))))

# ---------------- "true" hidden reward: 2*smooth - 1*rough ---------------
def r_star(traj):
    return 2.0 * traj[0] - 1.0 * traj[1]

rng = random.Random(42)

# 300 preference pairs: two random trajs per pair, human labels the better one
# by comparing hidden r_star (the learner NEVER sees r_star, only the label).
pairs = []
while len(pairs) < 300:
    a = (rng.uniform(0, 5), rng.uniform(0, 5))
    b = (rng.uniform(0, 5), rng.uniform(0, 5))
    ra, rb = r_star(a), r_star(b)
    if ra == rb:
        continue
    pairs.append((a, b) if ra > rb else (b, a))   # (win, lose)

print(f"생성된 선호 쌍: {len(pairs)}개")

생성된 선호 쌍: 300개


## 2. Bradley-Terry 모델로 보상 가중치 학습

보상모델 $r_\phi(\text{traj}) = w_1\cdot\text{traj}[0] + w_2\cdot\text{traj}[1] + b$를
선호 확률을 시그모이드로 모델링하는 Bradley-Terry 손실
$-\log\sigma(r_w - r_l)$로 최소화합니다.

**주의: $b$는 학습되지 않습니다.** Bradley-Terry 손실은 오직 두 궤적의
**차이** $r_w - r_l$에만 의존하는데, 공통 상수 $b$는 이 차이에서 완전히
소거되므로 추정 불가능합니다 — 그래서 $b$를 0으로 고정하고 $w_1, w_2$만
갱신합니다.

In [2]:
# ---------------- learn r_phi(t) = w1*t[0] + w2*t[1] + b -----------------
# NOTE: b is NOT learned. Bradley-Terry depends only on the difference
# r_w - r_l, in which any shared bias cancels exactly, so b is unidentifiable.
w1, w2, b = 0.0, 0.0, 0.0
lr = 0.01
history = []
for epoch in range(200):
    total = 0.0
    for win, lose in pairs:
        rw = w1 * win[0] + w2 * win[1] + b
        rl = w1 * lose[0] + w2 * lose[1] + b
        p = sigmoid(rw - rl)
        g = p - 1.0          # 정답 라벨은 항상 "win이 이겼다"(=1)
        total += -math.log(max(p, 1e-12))
        w1 -= lr * g * (win[0] - lose[0])
        w2 -= lr * g * (win[1] - lose[1])
        # b is deliberately not updated (cancels in r_w - r_l)
    history.append(total / len(pairs))

print(f"학습된 가중치: w1(부드러움)={w1:.3f}, w2(덜컹거림)={w2:.3f} (b=0, 학습되지 않음)")
print(f"진짜 가중치:   w1=2, w2=-1")
print(f"학습된 비율 w1/w2 = {w1 / w2:.2f}   (실제 비율 = -2)")

학습된 가중치: w1(부드러움)=7.264, w2(덜컹거림)=-3.643 (b=0, 학습되지 않음)
진짜 가중치:   w1=2, w2=-1
학습된 비율 w1/w2 = -1.99   (실제 비율 = -2)


## 3. 보지 못한 새 궤적 쌍 3개로 검증

학습에 쓰인 적 없는 3개의 새 쌍을 골라 "모델이 어느 쪽을 더 좋아한다고
예측하는가"를 확인합니다. 세 쌍 모두 실제 선호(부드러움이 높은 쪽)를
맞춰야 합니다.

In [3]:
# held-out pairs (never shown in training)
val = [((4.0, 1.0), (1.0, 4.0)),
       ((3.0, 3.0), (1.0, 1.0)),
       ((5.0, 0.0), (0.0, 5.0))]
for (s, r1), (s2, r2) in val:
    win = (max(r1, r2), min(r1, r2))
    lose = (min(r1, r2), max(r1, r2))
    rw = w1 * win[0] + w2 * win[1]
    rl = w1 * lose[0] + w2 * lose[1]
    pred = "앞쪽" if rw > rl else "뒤쪽"
    mark = "✓" if pred == "앞쪽" else "✗"
    print(f"(부드러움{s},덜컹거림{r1}) vs (부드러움{s2},덜컹거림{r2}): 모델 예측={pred} {mark}  (차이={rw - rl:.2f})")

(부드러움4.0,덜컹거림1.0) vs (부드러움1.0,덜컹거림4.0): 모델 예측=앞쪽 ✓  (차이=32.72)
(부드러움3.0,덜컹거림3.0) vs (부드러움1.0,덜컹거림1.0): 모델 예측=앞쪽 ✓  (차이=21.81)
(부드러움5.0,덜컹거림0.0) vs (부드러움0.0,덜컹거림5.0): 모델 예측=앞쪽 ✓  (차이=54.53)


## 4. 복원된 보상 방향 시각화

궤적을 (부드러움, 덜컹거림) 좌표로 그으면, 진짜 보상 $r^*$는 (2, −1)
방향의 등고면을 갖고, 학습된 보상모델 $r_\phi$는 (w1, w2) 방향의
등고면을 갖습니다. 두 방향 벡터는 *크기(스케일)는 달라도* 방향이 거의
일치합니다 — Bradley-Terry가 차이에만 의존하므로 스케일은 학습되지
않고, *방향(비율)만이* 학습된다는 것을 그림이 그대로 보여줍니다.

In [4]:
import numpy as np
fig, ax = plt.subplots(figsize=(5.4, 5.4))
xs = [t for t in range(-5, 7)]
X, Y = np.meshgrid(xs, xs)
Z = w1 * X + w2 * Y
cs = ax.contour(X, Y, Z, levels=14, colors="#adb5bd", linewidths=1.0)
ax.clabel(cs, fontsize=7, fmt="%.0f")
# true direction (2, -1) and learned direction (w1, w2)
ax.arrow(0, 0, 2, -1, head_width=0.45, head_length=0.35, fc="#1971c2", ec="#1971c2",
         lw=2.2, zorder=5)
ax.text(2.2, -1.5, "진짜\n$r^*$=(2,-1)", color="#1971c2", fontsize=10)
ax.arrow(0, 0, w1 * 0.55, w2 * 0.55, head_width=0.45, head_length=0.4,
         fc="#e8590c", ec="#e8590c", lw=2.2, zorder=5)
ax.text(w1 * 0.55 + 0.25, w2 * 0.55 + 0.5, f"학습된\n(${w1:.2f},{w2:.2f}$)",
        color="#e8590c", fontsize=10)
ax.scatter([4, 1, 5], [1, 4, 0], s=60, color="#51cf66", zorder=4)
ax.scatter([1, 4, 0], [4, 1, 5], s=60, color="#ff8787", zorder=4)
ax.set_xlabel("부드러움")
ax.set_ylabel("덜컹거림")
ax.set_title("2개 특징 궤적공간에서 복원된 보상 방향")
fig.savefig(IMG + "/ch12_2_reward_direction.svg", bbox_inches="tight")
plt.close(fig)
print("저장: ch12_2_reward_direction.svg")

저장: ch12_2_reward_direction.svg


## 5. 학습 곡선

300개 선호 쌍에 200 에폭 동안 Bradley-Terry 손실을 최소화하면 처음에는
급격히 떨어지다가 어느 시점부터 거의 수평이 됩니다 — 모델이 "부드러움이
더 가치 있다"는 방향을 *빠르게* 잡았다는 뜻입니다. (곡선이 0까지 안
내려가는 이유는, 서로 *가까운* 궤적 쌍은 진짜 보상에서도 차이가 작아
$\sigma(r_w-r_l)$가 1에 완전히 도달하지 못하기 때문 — 손실의 *절대값*
보다 *하향 추세*를 읽으세요.)

In [5]:
fig, ax = plt.subplots(figsize=(6.0, 3.0))
ax.plot(range(1, 201), history, color="#1971c2", lw=1.6)
ax.set_xlabel("epoch")
ax.set_ylabel("평균 Bradley-Terry 손실  −log σ(r_w − r_l)")
ax.set_title("선호 쌍 300개만으로의 학습 곡선 (lr=0.01)")
ax.grid(alpha=0.3)
fig.savefig(IMG + "/ch12_2_reward_loss_curve.svg", bbox_inches="tight")
plt.close(fig)
print("저장: ch12_2_reward_loss_curve.svg")

저장: ch12_2_reward_loss_curve.svg


## 6. 연습: 쌍 수와 seed를 바꿔 비율이 어떻게 흔들리는지

본문의 FAQ가 권하는 실험입니다. 선호 쌍 300개 → 30개로 줄이면 (부드러움,
덜컹거림) 평면의 *각 방향*이 고르게 커버되지 않아 학습된 비율 $w_1/w_2$가
−2에서 꽤 벗어날 수 있습니다. 쌍 수가 아니라 *쌍이 특징 공간의 서로 다른
방향을 얼마나 대표하느냐*가 핵심이라는 점을 체감해 보세요.

In [6]:
def train(n_pairs, seed, epochs=200):
    rng = random.Random(seed)
    ps = []
    while len(ps) < n_pairs:
        a = (rng.uniform(0, 5), rng.uniform(0, 5))
        bb = (rng.uniform(0, 5), rng.uniform(0, 5))
        ra, rb = r_star(a), r_star(bb)
        if ra == rb:
            continue
        ps.append((a, bb) if ra > rb else (bb, a))
    w1, w2 = 0.0, 0.0
    for epoch in range(epochs):
        for win, lose in ps:
            rw = w1 * win[0] + w2 * win[1]
            rl = w1 * lose[0] + w2 * lose[1]
            p = sigmoid(rw - rl)
            g = p - 1.0
            w1 -= lr * g * (win[0] - lose[0])
            w2 -= lr * g * (win[1] - lose[1])
    return w1 / w2 if w2 != 0 else float("nan")

for n in (300, 100, 30):
    ratios = [train(n, s) for s in (0, 1, 2, 3, 4)]
    print(f"쌍 {n:>3}개: 비율 w1/w2 = " + ", ".join(f"{r:.2f}" for r in ratios) + f"   (참값 -2.00)")

쌍 300개: 비율 w1/w2 = -1.93, -1.94, -1.88, -2.01, -2.04   (참값 -2.00)
쌍 100개: 비율 w1/w2 = -1.85, -2.04, -1.78, -1.94, -1.84   (참값 -2.00)
쌍  30개: 비율 w1/w2 = -1.95, -1.84, -1.34, -2.06, -1.92   (참값 -2.00)
